# Compare Segmentation: Cellpose vs Omnipose
Loads saved results and opens napari with additive-blended binary masks.
- **green** = cellpose post-QC
- **magenta** = omnipose
- **white/yellow** = overlap (agreement)
- **cyan** = omnipose long-cell flags (optional)

**Kernel:** `conda activate omnipose`

In [1]:
from pathlib import Path

# ── USER SETTINGS ────────────────────────────────────────────────────────────
_base         = Path.home() / "Box/Zohar_Persky/projects/p2f-revisions/morph-omnipose"
PHASE_DIR     = _base / "phase-proj"
CELLPOSE_DIR  = _base / "old_seg"
OMNIPOSE_DIR  = _base / "omnipose_seg"
LONG_CELL_DIR = _base / "long_cell_filter_by_length"

FOV = 1   # FOV number
HYB = 1   # hyb number
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
import tifffile

fov_name = f"fov_{FOV}_hyb_{HYB}"

phase_path    = PHASE_DIR     / f"{fov_name}.phase.tif"
cellpose_path = CELLPOSE_DIR  / fov_name / f"{fov_name}.seg.manual_qc.npy"
omnipose_path = OMNIPOSE_DIR  / fov_name / f"{fov_name}.seg.npy"
long_path     = LONG_CELL_DIR / fov_name / f"{fov_name}.seg.long_cells.npy"

phase    = tifffile.imread(phase_path)
cellpose = (np.load(cellpose_path) > 0).astype(np.uint8)
omnipose = (np.load(omnipose_path) > 0).astype(np.uint8)
long_cells = (np.load(long_path) > 0).astype(np.uint8) if long_path.exists() else None

print(f"FOV       : {fov_name}")
print(f"Phase     : {phase.shape} {phase.dtype}")
print(f"Cellpose  : {int(np.load(cellpose_path).max())} cells")
print(f"Omnipose  : {int(np.load(omnipose_path).max())} cells")
print(f"Long cells: {'found' if long_cells is not None else 'not found (skipped)'}")

FOV       : fov_1_hyb_1
Phase     : (2048, 2048) uint16
Cellpose  : 8304 cells
Omnipose  : 11796 cells
Long cells: found


In [3]:
import napari

viewer = napari.Viewer(title=f"Compare segmentation — {fov_name}")

viewer.add_image(phase,     name="phase",
                 colormap="gray",    blending="translucent",
                 contrast_limits=[int(phase.min()), int(phase.max())])
viewer.add_image(cellpose,  name="cellpose (green)",
                 colormap="green",   blending="additive",  opacity=0.6,
                 contrast_limits=[0, 1])
viewer.add_image(omnipose,  name="omnipose (magenta)",
                 colormap="magenta", blending="additive",  opacity=0.6,
                 contrast_limits=[0, 1])

if long_cells is not None:
    viewer.add_image(long_cells, name="long cells — omnipose (cyan)",
                     colormap="cyan", blending="additive", opacity=0.7,
                     contrast_limits=[0, 1])

napari.run()